In [10]:
# !pip install python-dotenv requests pandas

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

DOCUMENT_ENDPOINT = os.getenv("DOCUMENT_ENDPOINT")
DOCUMENT_KEY = os.getenv("DOCUMENT_KEY")

ML_ENDPOINT = os.getenv("ML_ENDPOINT")
ML_KEY = os.getenv("ML_KEY")

SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_KEY = os.getenv("SEARCH_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")

print("Environment loaded")

Environment loaded


In [12]:
import requests
import time

def extract_text_from_image(image_path):

    url = f"{DOCUMENT_ENDPOINT}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

    headers = {
        "Content-Type": "application/octet-stream",
        "Ocp-Apim-Subscription-Key": DOCUMENT_KEY
    }

    with open(image_path, "rb") as f:
        response = requests.post(url, headers=headers, data=f)

    operation_url = response.headers["operation-location"]

    while True:

        result = requests.get(
            operation_url,
            headers={"Ocp-Apim-Subscription-Key": DOCUMENT_KEY}
        )

        result_json = result.json()

        if result_json["status"] == "succeeded":
            break

        time.sleep(2)

    text = ""

    for page in result_json["analyzeResult"]["pages"]:
        for line in page["lines"]:
            text += line["content"] + "\n"

    return text

In [13]:
import re

def extract_fields(text):

    data = {}

    patterns = {
        "Customer_ID": r"Customer ID[:\s]+([A-Z0-9]+)",
        "Claim_Amount": r"Claim Amount[:\s]+(\d+)",
        "Claim_Type": r"Claim Type[:\s]+([A-Za-z]+)",
        "Location": r"Location[:\s]+([A-Za-z]+)",
        "Previous_Claims": r"Previous Claims[:\s]+(\d+)|Number of Previous Claims[:\s]+(\d+)"
    }

    for field, pattern in patterns.items():

        match = re.search(pattern, text, re.IGNORECASE)

        if match:

            value = next(g for g in match.groups() if g)

            if field in ["Claim_Amount", "Previous_Claims"]:
                value = int(value)

            data[field] = value

    return data

In [14]:
def predict_fraud(data):

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {ML_KEY}"
    }

    body = {
        "Inputs": {
            "input1":[data]
        },
        "GlobalParameters": {}
    }

    response = requests.post(
        ML_ENDPOINT,
        headers=headers,
        json=body
    )

    return response.json()

In [15]:
def get_prediction_label(result):

    prediction = result["Results"]["WebServiceOutput0"][0]["Scored Labels"]

    probability = result["Results"]["WebServiceOutput0"][0]["Scored Probabilities"]

    return prediction, probability

In [16]:
def upload_to_search(data, prediction):

    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version=2021-04-30-Preview"

    headers = {
        "Content-Type": "application/json",
        "api-key": SEARCH_KEY
    }

    document = {
        "Customer_ID": data["Customer_ID"],
        "Claim_Amount": data["Claim_Amount"],
        "Claim_Type": data["Claim_Type"],
        "Location": data["Location"],
        "Previous_Claims": data["Previous_Claims"],
        "FraudPrediction": str(prediction)
    }

    body = {
        "value":[
            {
                "@search.action":"upload",
                **document
            }
        ]
    }

    response = requests.post(url, headers=headers, json=body)

    return response.json()

In [17]:
def search_claims(query):

    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2021-04-30-Preview"

    headers = {
        "Content-Type": "application/json",
        "api-key": SEARCH_KEY
    }

    body = {
        "search": query
    }

    response = requests.post(url, headers=headers, json=body)

    return response.json()

In [21]:
import os

image_folder = "images"

for file in os.listdir(image_folder):

    path = f"{image_folder}/{file}"

    print("\nProcessing:", file)

    # 1 extract text
    text = extract_text_from_image(path)

    # 2 convert text → structured json
    structured_data = extract_fields(text)

    print("Extracted:", structured_data)

    # skip incomplete extraction
    if len(structured_data) < 5:
        print("Skipping incomplete data")
        continue

    # 3 send to ML model
    prediction_result = predict_fraud(structured_data)

    prediction, probability = get_prediction_label(prediction_result)

    print("Fraud:", prediction)
    print("Confidence:", probability)

    # 4 STORE RESULT IN AI SEARCH  ← THIS IS IMPORTANT
    upload_to_search(structured_data, prediction)

    print("Stored in AI Search")


Processing: img1.png
Extracted: {'Customer_ID': 'C002', 'Claim_Amount': 75000, 'Claim_Type': 'Accident', 'Location': 'Mumbai', 'Previous_Claims': 2}
Fraud: True
Confidence: 0.9030325759632543
Stored in AI Search

Processing: img2.png
Extracted: {'Customer_ID': 'C003', 'Claim_Amount': 64000, 'Claim_Type': 'Theft', 'Location': 'Bangalore', 'Previous_Claims': 1}
Fraud: True
Confidence: 0.7893302269536135
Stored in AI Search

Processing: img3.png
Extracted: {'Customer_ID': 'C001', 'Claim_Amount': 75000, 'Claim_Type': 'Medical', 'Location': 'Delhi', 'Previous_Claims': 2}
Fraud: False
Confidence: 0.25351525552457965
Stored in AI Search


In [ ]:
result = search_claims("Delhi")
print(result)

{'@odata.context': "https://insurancesearchv1.search.windows.net/indexes('claims-index')/$metadata#docs(*)", 'value': [{'@search.score': 0.6931472, 'Customer_ID': 'C001', 'Claim_Amount': 75000, 'Claim_Type': 'Medical', 'Location': 'Delhi', 'Previous_Claims': 2, 'FraudPrediction': 'False'}]}


In [23]:
upload_to_search(structured_data, prediction)

{'@odata.context': "https://insurancesearchv1.search.windows.net/indexes('claims-index')/$metadata#Collection(Microsoft.Azure.Search.V2021_04_30_Preview.IndexResult)",
 'value': [{'key': 'C001',
   'status': True,
   'errorMessage': None,
   'statusCode': 200}]}

In [ ]:
search_claims("False")

{'@odata.context': "https://insurancesearchv1.search.windows.net/indexes('claims-index')/$metadata#docs(*)",
 'value': [{'@search.score': 0.47000363,
   'Customer_ID': 'C001',
   'Claim_Amount': 75000,
   'Claim_Type': 'Medical',
   'Location': 'Delhi',
   'Previous_Claims': 2,
   'FraudPrediction': 'False'}]}

In [25]:
search_claims("Delhi")

{'@odata.context': "https://insurancesearchv1.search.windows.net/indexes('claims-index')/$metadata#docs(*)",
 'value': [{'@search.score': 0.47000363,
   'Customer_ID': 'C001',
   'Claim_Amount': 75000,
   'Claim_Type': 'Medical',
   'Location': 'Delhi',
   'Previous_Claims': 2,
   'FraudPrediction': 'False'}]}

In [26]:
# VIVA Question

In [ ]:
# 1️⃣ What is Semantic Search?

# Semantic search understands the meaning of the query, not just exact keywords.

# Normal search → matches words
# Semantic search → understands intent

# Example

# Search:

# high claim cases

# Even if document contains:

# Claim Amount = 90000

# Semantic search still finds it because it understands high = large amount.

# In your project

# User searches:

# fraud cases in Delhi

# Azure AI Search returns relevant documents based on meaning.

In [ ]:
# 2️⃣ Difference between Indexing and Querying
# Indexing	Querying
# storing data in search engine	searching data
# happens once when data added	happens when user searches
# prepares data for fast search	retrieves matching results
# example: upload claim data	example: search "Delhi claims"
# Simple explanation

# Indexing = saving data
# Querying = finding data

In [ ]:
# 3️⃣ What is Model Drift?

# Model drift happens when real-world data changes over time, so model accuracy decreases.

# Example

# Your fraud model trained on old data:

# Fraud mostly occurs when:

# Claim Amount > 50000

# After 1 year:
# Fraud patterns change.

# Now fraud happens even at:

# Claim Amount = 20000

# Model performance becomes poor → this is model drift.

# Solution

# Retrain model regularly using new data.

In [ ]:
# 4️⃣ Why versioning is important?

# Versioning means saving different versions of model.

# Benefits

# • track changes in model
# • rollback if new model is worse
# • compare performance
# • maintain reproducibility

# Example

# Model v1 accuracy = 82%
# Model v2 accuracy = 90%

# Use v2.

# If v3 gives 70%, rollback to v2.

In [ ]:
# 5️⃣ How APIs help integration?

# API allows different services to communicate automatically.

# In your project APIs connect:

# Document Intelligence API
# ↓
# ML prediction API
# ↓
# Azure AI Search API

# Example flow

# Image uploaded
# ↓
# API extracts text
# ↓
# API sends data to ML model
# ↓
# API stores result in AI Search
# ↓
# User searches using API

# Everything works automatically without manual work.